# Trabalho sobre LLM

## Equipe 8

- Gleice Souza
- Lucas Batista
- Matheus Mendes

In [1]:
import time
import metrics
import ollama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from format_output import gerar_markdown_por_pergunta
from IPython.display import display, Markdown
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
model_judge = "llama3.1:8b"

In [3]:
list_models = [model.model for model in ollama.list().models if model.model not in (model_judge, "nomic-embed-text:latest")]
list_models

['deepseek-r1:14b', 'gemma3:12b', 'phi4:14b', 'qwen3.5:9b']

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma(
    persist_directory="./data_documents",
    embedding_function=embeddings
)

# preparação para a pergunta com similaridade de 3 trechos mais relevantes parecidos
# Configura o buscador para trazer os 3 blocos mais relevantes para a pergunta
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [5]:
template_rag = """
[CONTEXTO]
Utilize estritamente as informações extraídas do regulamento de estágio da UFRB em PDF fornecidas abaixo para responder à pergunta do usuário de forma clara e direta.
Se a resposta não puder ser encontrada nos fragmentos, diga apenas que a informação não consta no documento.

FRAGMENTOS DO PDF:
{context}

[PERGUNTA]
{question}

[RESPOSTA]
"""

In [6]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def chat(model_name, pergunta):
    llm = ChatOllama(model=model_name, temperature=0)
    
    prompt = ChatPromptTemplate.from_template(template_rag)
    
    # Construção da cadeia RAG
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain.invoke(pergunta)

In [7]:
qa_pairs = [
    {
        "pergunta": "Quais documentos são necessários para redução de carga horária?",
        "resposta_correta": """I-Comprovante de vínculo empregatício (cópia da Carteira de Trabalho ou cópia de nomeação no Diário Oficial);
II - Três últimos contracheques (apenas a parte que indica nome, matrícula e mês do pagamento);
III - Atestado de frequência da escola, discriminando nível de ensino, ano, disciplina, turno e carga horária;
IV - Relatório da Coordenação de Área, ou Coordenação Pedagógica ou da Direção, avaliando o perfil profissional do professor em formação.""",
    },
    {
        "pergunta": "Pode ser solicitado dispensa do estágio supervisionado obrigatório?",
        "resposta_correta": """Sob nenhuma hipótese o estudante será dispensado do Estágio Supervisionado
Obrigatório, nem mesmo será permitida a realização de atividades domiciliares por
motivo de doença ou licença maternidade. Nestes casos, o estudante não poderá se
matricular. Entretanto, caso ele esteja cursando o componente curricular Estágio
Supervisionado deverá solicitar o trancamento do mesmo e se matricular em outro
semestre, no prazo estipulado pela Universidade.""",
    },
]

In [8]:
results = []

In [9]:
def eval_question(model, question, correct_answer):
    
    inicio = time.time()
    final_answer = chat(model, question)
    fim = time.time()

    metricas = metrics.calcular_metricas_operacionais(
        prompt=question,
        resposta=final_answer,
        tempo_inicio=inicio,
        tempo_fim=fim
    )

    rouge = metrics.calcular_rouge(final_answer, correct_answer)
    
    eval_judge = metrics.llmAsJudge(question, final_answer, correct_answer, model_judge)

    return {
        "Pergunta": question, 
        "Modelo": model,
        "Resposta final": final_answer,
        "Resposta correta": correct_answer,
        "Tokens": metricas["tokens_totais"],
        "Latência (s)": metricas["latencia_segundos"],
        "Tokens/s": metricas["tokens_por_segundo"],
        "Custo (USD)": metricas["custo_estimado_usd"],
        "ROUGE-1": rouge["rouge1_f1"],
        "ROUGE-2": rouge["rouge2_f1"],
        "ROUGE-L": rouge["rougeL_f1"],
        "Faithfulness": eval_judge["faithfulness"]
    }

In [10]:
for qa in qa_pairs:
    print(f"\nAvaliando a pergunta: {qa["pergunta"]}")

    for model in list_models:
        
        print(f"\nAvaliando modelo: {model}")

        results.append(eval_question(model, qa["pergunta"], qa["resposta_correta"]))
        
        print(75*"=")
    
    print(75*"*")


Avaliando a pergunta: Quais documentos são necessários para redução de carga horária?

Avaliando modelo: deepseek-r1:14b


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]


Avaliando modelo: gemma3:12b


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]


Avaliando modelo: phi4:14b


...


Avaliando modelo: phi4:14b


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]


Avaliando modelo: qwen3.5:9b


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

***************************************************************************


In [11]:
results

[{'Pergunta': 'Quais documentos são necessários para redução de carga horária?',
  'Modelo': 'deepseek-r1:14b',
  'Resposta final': 'Para a redução de carga horária no estágio na UFRB, são necessários os seguintes documentos:\n\n1. Formulário de Requisitos para Redução de Carga Horária  \n2. Projeto de Estágio  \n3. Cronograma de Atividades  \n4. Relatório de Orientador  \n5. Declaração de Compromisso',
  'Resposta correta': 'I-Comprovante de vínculo empregatício (cópia da Carteira de Trabalho ou cópia de nomeação no Diário Oficial);\nII - Três últimos contracheques (apenas a parte que indica nome, matrícula e mês do pagamento);\nIII - Atestado de frequência da escola, discriminando nível de ensino, ano, disciplina, turno e carga horária;\nIV - Relatório da Coordenação de Área, ou Coordenação Pedagógica ou da Direção, avaliando o perfil profissional do professor em formação.',
  'Tokens': 93,
  'Latência (s)': 61.26,
  'Tokens/s': 1.29,
  'Custo (USD)': 0.00014,
  'ROUGE-1': 0.2774,
  

In [12]:
display(Markdown(gerar_markdown_por_pergunta(results)))

### Quais documentos são necessários para redução de carga horária?

| Modelo | Latência (s) | Tokens | Tokens/s | Custo (USD) | ROUGE-1 | ROUGE-2 | ROUGE-L | Faithfulness |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| deepseek-r1:14b | 61.2600 | 93 | 1.2900 | 0.000140 | **0.2774** | **0.0593** | 0.1752 | 0.1667 |
| gemma3:12b | **12.6500** | 35 | 1.6600 | 0.000053 | 0.2075 | 0.0577 | 0.1321 | 0.0000 |
| phi4:14b | 22.9100 | 90 | **3.3200** | 0.000135 | 0.2466 | 0.0417 | **0.1918** | 0.0000 |
| qwen3.5:9b | 46.5300 | **23** | 0.1900 | **0.000034** | 0.1064 | 0.0000 | 0.0638 | **1.0000** |


### Pode ser solicitado dispensa do estágio supervisionado obrigatório?

| Modelo | Latência (s) | Tokens | Tokens/s | Custo (USD) | ROUGE-1 | ROUGE-2 | ROUGE-L | Faithfulness |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| deepseek-r1:14b | 60.9900 | 54 | 0.6400 | 0.000081 | **0.3469** | **0.1458** | 0.2245 | **0.0000** |
| gemma3:12b | **6.4100** | **24** | 1.4000 | **0.000036** | 0.1282 | 0.0263 | 0.1282 | **0.0000** |
| phi4:14b | 11.7100 | 60 | **3.8400** | 0.000090 | 0.3048 | 0.1165 | **0.2286** | **0.0000** |
| qwen3.5:9b | 78.7000 | **24** | 0.1100 | **0.000036** | 0.1282 | 0.0263 | 0.1282 | **0.0000** |
